In [ ]:
import json
import os
import time

import joblib
import numpy as np
import pandas as pd
from scipy.sparse import hstack
from sentence_transformers import SentenceTransformer
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

here = os.getcwd()
root = os.path.dirname(here)

seed = 42
test_size = 0.2
data_path = os.path.join(root, "data", "filtered_dataset.csv")
results_dir = os.path.join(here, "results")
saved_dir = os.path.join(results_dir, "saved")
os.makedirs(saved_dir, exist_ok=True)
print("data  :", data_path)
print("output:", results_dir)

In [ ]:
# load + clean: drop empties and case-insensitive duplicate prompts
df = pd.read_csv(data_path)[["prompt", "class"]].dropna()
df["prompt"] = df["prompt"].astype(str)
df = df[df["prompt"].str.strip() != ""]
df = df.loc[~df["prompt"].str.strip().str.lower().duplicated()].reset_index(drop=True)
df["class"] = df["class"].astype(int)

X = df["prompt"].to_numpy()
y = df["class"].to_numpy()
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=test_size, random_state=seed, stratify=y)

print(f"train: {len(X_train)}  test: {len(X_test)}")
print(f"train positives: {int(y_train.sum())}/{len(y_train)}  "
      f"test positives: {int(y_test.sum())}/{len(y_test)}")

In [ ]:
encoder = SentenceTransformer("all-MiniLM-L6-v2")


def embed(texts):
    return encoder.encode([str(t) for t in texts], batch_size=64,
                          convert_to_numpy=True, show_progress_bar=False).astype("float32")


emb_train = embed(X_train)
emb_test = embed(X_test)
print("embeddings:", emb_train.shape, emb_test.shape)

In [8]:
word_vectorizer = TfidfVectorizer(
    analyzer="word", ngram_range=(1, 2), min_df=3, max_df=0.9,
    max_features=30000, sublinear_tf=True, strip_accents="unicode", lowercase=True,
)
char_vectorizer = TfidfVectorizer(
    analyzer="char_wb", ngram_range=(3, 5), min_df=5,
    max_features=30000, sublinear_tf=True, lowercase=True,
)


def tfidf_fit_transform(texts):
    return hstack([word_vectorizer.fit_transform(texts),
                   char_vectorizer.fit_transform(texts)]).tocsr()


def tfidf_transform(texts):
    return hstack([word_vectorizer.transform(texts),
                   char_vectorizer.transform(texts)]).tocsr()

In [12]:
tfidf_train = tfidf_fit_transform(X_train)
svd = TruncatedSVD(n_components=200, random_state=seed)
lsa_train = svd.fit_transform(tfidf_train)
lsa_test = svd.transform(tfidf_transform(X_test))

In [13]:
X_train_features = np.hstack([emb_train, lsa_train]).astype("float32")
X_test_features = np.hstack([emb_test, lsa_test]).astype("float32")

In [ ]:
model = make_pipeline(
    StandardScaler(),
    SVC(kernel="rbf", C=10.0, gamma="scale", class_weight="balanced", random_state=seed),
)
model.fit(X_train_features, y_train)

y_pred = model.predict(X_test_features)
y_score = model.decision_function(X_test_features)
cm = confusion_matrix(y_test, y_pred, labels=[0, 1])

results = {
    "accuracy": round(float(accuracy_score(y_test, y_pred)), 4),
    "precision": round(float(precision_score(y_test, y_pred, zero_division=0)), 4),
    "recall": round(float(recall_score(y_test, y_pred, zero_division=0)), 4),
    "f1": round(float(f1_score(y_test, y_pred, zero_division=0)), 4),
    "roc_auc": round(float(roc_auc_score(y_test, y_score)), 4),
    "confusion_matrix": cm.tolist(),
}

[[2332   15]
 [  14 2071]]


In [15]:
models_dir = os.path.join(here, "models")
os.makedirs(models_dir, exist_ok=True)

joblib.dump(model, os.path.join(models_dir, "model_svm.joblib"))
joblib.dump({"word": word_vectorizer, "char": char_vectorizer, "svd": svd},
            os.path.join(models_dir, "featurizer.joblib"))

['c:\\Users\\Bakr\\Desktop\\role_playing\\Approach#11_HybridNoHandcrafted\\models\\featurizer.joblib']